# 🚀 DigiShakti CAPTCHA Neural Network Trainer (PyTorch + ONNX)
This notebook trains a high-precision **CRNN + CTC Loss Neural Network** on your 200 DigiShakti CAPTCHAs and exports `digishakti_captcha.onnx`.

In [ ]:
# 1. Upload captcha_dataset.zip
from google.colab import files
import zipfile
import os

print("Please upload captcha_dataset.zip...")
uploaded = files.upload()

if 'captcha_dataset.zip' in uploaded:
    with zipfile.ZipFile('captcha_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print("✅ Dataset extracted successfully!")
else:
    print("⚠️ Zip file not found. Please upload captcha_dataset.zip.")

In [ ]:
# 2. PyTorch Imports & Setup
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

CHARS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR_TO_INT = {char: i + 1 for i, char in enumerate(CHARS)}
INT_TO_CHAR = {i + 1: char for i, char in enumerate(CHARS)}
NUM_CLASSES = len(CHARS) + 1

In [ ]:
# 3. Dataset & Preprocessing Pipeline
class CaptchaDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_paths = glob.glob(os.path.join(img_dir, '*.*'))
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        filename = os.path.basename(path).split('.')[0].split('_')[0].upper()
        image = Image.open(path).convert('L')
        if self.transform:
            image = self.transform(image)
        label = [CHAR_TO_INT[c] for c in filename if c in CHAR_TO_INT]
        return image, torch.tensor(label, dtype=torch.long)

transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = CaptchaDataset('dataset', transform=transform)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"Loaded {len(dataset)} training images.")

In [ ]:
# 4. CRNN Neural Network Model Architecture
class CRNN(nn.Module):
    def __init__(self, num_classes=37):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d((2, 1), (2, 1))
        )
        self.rnn = nn.LSTM(256 * 2, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        features = self.cnn(x)
        b, c, h, w = features.size()
        features = features.permute(0, 3, 1, 2).reshape(b, w, c * h)
        out, _ = self.rnn(features)
        logits = self.fc(out)
        return logits

model = CRNN(NUM_CLASSES).to(device)
print("Model initialized successfully!")

In [ ]:
# 5. Train Model with CTC Loss
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 35
print(f"Training CRNN for {epochs} epochs...")

model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for imgs, labels in dataloader:
        imgs = imgs.to(device)
        optimizer.zero_grad()
        
        preds = model(imgs) # [B, W, C]
        log_probs = F.log_softmax(preds, dim=2).permute(1, 0, 2) # [W, B, C]
        
        input_lengths = torch.full((imgs.size(0),), preds.size(1), dtype=torch.long)
        target_lengths = torch.full((imgs.size(0),), 5, dtype=torch.long)
        
        loss = criterion(log_probs, labels, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{epochs}] - Loss: {total_loss/len(dataloader):.4f}")

print("🎉 Training complete!")

In [ ]:
# 6. Export Trained Model to ONNX
model.eval()
dummy_input = torch.randn(1, 1, 32, 128).to(device)
onnx_filename = "digishakti_captcha.onnx"

torch.onnx.export(
    model, dummy_input, onnx_filename,
    export_params=True, opset_version=12,
    do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"✅ Model successfully exported to {onnx_filename}!")
files.download(onnx_filename)